In [35]:
import pandas as pd
import numpy as np

In [36]:
df=pd.DataFrame({
    'Age':[20,30,25,None,40,None,23,None],
    'Salary':[30000,45000,None,60000,None,50000,None,55000],
    'Gender':['Male','Female',None,'Male',None,'Female','Male',None],
    'Job':['ML Engineer',np.nan,'Software Engineer',np.nan,'Frontend Developer','ML Engineer',np.nan,'Backend Developernan']
})

In [37]:
df

,Age,Salary,Gender,Job
0,20.0,30000.0,Male,ML Engineer
1,30.0,45000.0,Female,NaN
2,25.0,NaN,None,Software Engineer
3,NaN,60000.0,Male,NaN
4,40.0,NaN,None,Frontend Developer
5,NaN,50000.0,Female,ML Engineer
6,23.0,NaN,Male,NaN
7,NaN,55000.0,None,Backend Developernan


In [38]:
from sklearn.impute import SimpleImputer

In [39]:
imputer=SimpleImputer(strategy='constant',fill_value='unknown')

In [40]:
df['Gender']

0      Male
1    Female
2      None
3      Male
4      None
5    Female
6      Male
7      None
Name: Gender, dtype: object

In [41]:
imputer.fit_transform(df[['Gender','Job']])

array([['Male', 'ML Engineer'],
       ['Female', 'unknown'],
       [None, 'Software Engineer'],
       ['Male', 'unknown'],
       [None, 'Frontend Developer'],
       ['Female', 'ML Engineer'],
       ['Male', 'unknown'],
       [None, 'Backend Developernan']], dtype=object)

In [42]:
imputer=SimpleImputer(strategy='mean')

In [43]:
imputer.fit_transform(df[['Salary','Age']])

array([[3.00e+04, 2.00e+01],
       [4.50e+04, 3.00e+01],
       [4.80e+04, 2.50e+01],
       [6.00e+04, 2.76e+01],
       [4.80e+04, 4.00e+01],
       [5.00e+04, 2.76e+01],
       [4.80e+04, 2.30e+01],
       [5.50e+04, 2.76e+01]])

In [44]:
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [45]:
df1=pd.read_csv('Instagram visits clustering.csv')

In [46]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   User ID                  2600 non-null   int64  
 1   Instagram visit score    2600 non-null   int64  
 2   Spending_rank(0 to 100)  2600 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 61.1 KB


In [47]:
df1.head()

,User ID,Instagram visit score,Spending_rank(0 to 100)
0,0,63,24.050708
1,1,61,25.223290
2,2,104,18.528245
3,3,82,86.890232
4,4,14,31.492397


In [48]:
df1.drop(columns=['User ID'],inplace=True)

In [54]:
standard_col=['Instagram visit score','Spending_rank(0 to 100)']

In [55]:
preprocessing=ColumnTransformer(
    transformers=[
        ('standardsclaer',StandardScaler(),standard_col)
    ],remainder='passthrough'
)

In [56]:
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',KMeans(n_clusters=2,random_state=42))
    ]
)

In [57]:
main_pipeline.fit(df1)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standardsclaer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [58]:
new_customer=pd.DataFrame({
    'Instagram visit score': [100],
    'Spending_rank(0 to 100)': [200]
})

In [59]:
main_pipeline.predict(new_customer)

array([1], dtype=int32)